# Chapter 8 – Worked examples

This notebook runs the examples of [Chapter 8](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-2-circuits/ch08-dc-circuits/) live. The code lives in the modules next to it
(`nodal.py`, `wheatstone.py`); the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q numpy

In [ ]:
import nodal
import numpy as np
import wheatstone

## Two nodes plus ground

Conductances 1 S (node 1 to ground), 2 S (nodes 1–2), 4 S (node 2 to ground), a 3 A source into node 1:
$G v = i$ with $G = A^\top \operatorname{diag}(g) A$ minus the ground row and column gives
$v_1 = 18/14$, $v_2 = 6/14$.

In [ ]:
a = nodal.incidence(nodal.EX_FR, nodal.EX_TO, 3)
print('A =\n', a)
print('A^T diag(g) A =\n', a.T @ np.diag(nodal.EX_G) @ a, '\n(delete row and column 0 to get G)')
v, i = nodal.nodal_solve(nodal.EX_FR, nodal.EX_TO, nodal.EX_G, nodal.EX_INJ, 3)
print('v =', np.round(v, 4), ' 18/14 =', round(18 / 14, 4), ' 6/14 =', round(6 / 14, 4))
print('branch currents (1-0, 1-2, 2-0) =', np.round(i, 4))

## The Chapter 2 network as a resistor circuit

Six nodes, seven branches, 0.5 A into node 2 and 1 A out of node 5; ground supplies the other 0.5 A.
KCL holds at every node and the source power equals the Joule power in the branches.

In [ ]:
v, i = nodal.nodal_solve(nodal.FR, nodal.TO, nodal.G, nodal.INJ, nodal.N)
for k, (f, t) in enumerate(zip(nodal.FR, nodal.TO, strict=True)):
    print(f'branch ({f}, {t}): {nodal.G[k]:4.0f} S  current {i[k]:+.5f} A')
print('v =', np.round(v, 5))
print('KCL residual:', np.round(nodal.kcl_residual(nodal.FR, nodal.TO, i, nodal.INJ, nodal.N), 12))
sources, branches = nodal.power_balance(v, i, nodal.INJ, nodal.G)
print(f'power: sources {sources:.6f} W, branches {branches:.6f} W')

## Thévenin resistance = effective resistance

Inject 1 A at node 5, take it out at node 0: the voltage difference is the Thévenin resistance, equal to the
effective resistance $(e_5 - e_0)^\top L^+ (e_5 - e_0)$ of Chapter 3.

In [ ]:
r_th = nodal.thevenin_resistance(nodal.FR, nodal.TO, nodal.G, nodal.N, 5, 0)
r_eff = nodal.effective_resistance(nodal.FR, nodal.TO, nodal.G, nodal.N, 5, 0)
print(f'Thevenin {r_th:.5f} ohm, effective resistance {r_eff:.5f} ohm')

## Splitting a node: the coupler carries the imbalance of busbar A

Node 2 becomes busbars A (branches (0, 2), (1, 2), the 0.5 A) and B (branch (2, 4)), joined by a coupler.
With a near-ideal coupler its current equals the KCL imbalance of A computed from the unsplit solution; a
weak coupler departs from it.

In [ ]:
for cg in (1e6, 1e3, 10.0):
    print(cg, nodal.split_node_2(cg))

## A balanced Wheatstone bridge

$R_1 R_4 = R_2 R_3$ makes the two corner voltages equal and the middle branch current zero.

In [ ]:
for r in ([100.0, 200.0, 50.0, 100.0, 1000.0], [100.0, 200.0, 50.0, 120.0, 1000.0]):
    v, i = wheatstone.bridge_currents(r)
    print(r, 'balanced' if wheatstone.is_balanced(r) else 'unbalanced', f'middle current {i[4]:+.6f} A')